# HVAC v19 retrain — Colab via Drive link (no upload widget, no phone)

Use this when the **file-upload widget fails** (reCAPTCHA / stuck). Instead, Colab
**downloads** the bundle from a Google Drive **share link** — fast and reliable.

### One-time setup (in Google Drive, in your browser)
1. Make sure **`hvac_v19s_tiled_colab.zip`** (~940 MB) is in your Drive and fully
   uploaded (green check). If the big folder upload didn't finish it, upload just
   that ONE file to Drive now (a single-file upload to Drive is reliable).
2. Right-click that zip -> **Share -> General access -> "Anyone with the link" -> Copy link.**
3. Paste the link into `LINK` in cell 3 below.

Then: `Runtime -> Change runtime type -> GPU`, `Runtime -> Run all`.
At the end it downloads `hvac_yolov8s_v19.pt` to your computer.

In [ ]:
# 1. GPU check
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NONE -> Runtime > Change runtime type > GPU, then Run all again')

In [ ]:
# 2. Install
!pip -q install ultralytics==8.4.51 gdown

In [ ]:
# 3. Download the bundle from your Drive share link, then unzip.
#    PASTE your 'Anyone with the link' URL for hvac_v19s_tiled_colab.zip below.
LINK = 'PASTE_DRIVE_SHARE_LINK_HERE'
import gdown, zipfile, os
assert LINK.startswith('http'), 'Paste the Drive share link into LINK first.'
gdown.download(url=LINK, output='bundle.zip', fuzzy=True, quiet=False)
sz = os.path.getsize('bundle.zip') / 1e6
print(f'downloaded bundle.zip = {sz:.0f} MB')
assert sz > 500, 'File looks too small - Drive may have returned an HTML page, not the zip. Check sharing is "Anyone with the link".'
os.makedirs('/content/hvac', exist_ok=True)
with zipfile.ZipFile('bundle.zip') as z:
    z.extractall('/content/hvac')
os.chdir('/content/hvac')
print('ready:', os.getcwd(), sorted(os.listdir('.'))[:8])

In [ ]:
# 4. Train (v10 base -> v19). ~1-1.5h on a T4 (lower --epochs to 30 for a faster first run).
!python train_v11.py --data-yaml yolo_dataset_v19s_tiled/data.yaml \
    --epochs 60 --device 0 --batch 16 --imgsz 640 --run-name v19

In [ ]:
# 5. Download the trained model
import glob
from google.colab import files
cands = glob.glob('models/hvac_yolov8s_v19.pt') + glob.glob('runs/**/weights/best.pt', recursive=True)
assert cands, 'No weights found - check the training cell output.'
print('downloading', cands[0])
files.download(cands[0])